# 10 · Validate Retrieval on Actual Course PDFs

This notebook ingests the course archive and tests five source-title-focused queries across clinical and food security. The evaluation is *not* on fabricated documents. Its simple target-title measure is not a substitute for expert-reviewed relevance labels, and 2025 SEC or full product documentation is not present. Retrieval remains local TF-IDF/BM25 unless you separately configure and index a real embedding provider.

In [1]:
# Execute course ingestion if this project database has not yet indexed course reports.
%run 03_evidence_recommendations.ipynb
%run 09_course_document_import.ipynb
from time import perf_counter
from collections import Counter

CASES_REAL=[
 ('csr', 'What does the 2023 SOFI report explain about urbanization and healthy diets across the rural urban continuum?', 'SOFI-2023.pdf'),
 ('csr', 'How does the SOFI 2024 report discuss financing to end hunger and malnutrition?', 'SOFI-2024.pdf'),
 ('clinical', 'How are machine learning methods applied to heart attack prediction?', 'Machine_Learning_Approaches_for_Predicting_Heart_Attacks.pdf'),
 ('clinical', 'What role can mobile app based patient education play in improving self management of type 2 diabetes?', 'Improving_Self-Management_of_Type_2_Diabetes'),
 ('clinical', 'What design and control methods are described for a powered wearable knee exoskeleton used in rehabilitation?', 'Design_and_control_of_a_novel_powered_wearable_knee_exoskeleton'),
]

def evaluate_actual_course(k=5):
    report=[]
    for domain,question,expected in CASES_REAL:
        start=perf_counter()
        hits=hybrid_search(PROJECT_ID,question,domain,top_k=k,exclude_synthetic=True)
        ms=round((perf_counter()-start)*1000,2)
        expected_rank=next((j+1 for j,h in enumerate(hits) if expected.casefold() in h['filename'].casefold()),None)
        report.append({'domain':domain,'question':question,'target_title_fragment':expected,
                       'target_rank':expected_rank,'latency_ms':ms,
                       'citation_validity':all(citation_exists(PROJECT_ID,h['chunk_id']) for h in hits),
                       'hits':[{'citation':h['citation'],'source_type':h['source_type'],
                                'score':h['score'],'excerpt':h['excerpt'][:330]} for h in hits[:3]]})
    found=sum(x['target_rank'] is not None for x in report)
    aggregate={'dataset':'ACTUAL COURSE-SUPPLIED PDF FILES','cases':len(report),
               'target_title_recall_at_5':round(found/len(report),3),
               'mean_retrieval_latency_ms':round(sum(x['latency_ms'] for x in report)/len(report),2),
               'all_returned_citations_resolve':all(x['citation_validity'] for x in report),
               'note':'Small hand-selected title-focused smoke test, not expert relevance judgments or validated accuracy.',
               'details':report}
    return aggregate

course_eval=evaluate_actual_course()
print(json.dumps({k:v for k,v in course_eval.items() if k!='details'},indent=2))
for case in course_eval['details']:
    print('\nDOMAIN:',case['domain'],'EXPECTED:',case['target_title_fragment'],'RANK:',case['target_rank'])
    for hit in case['hits']:
        print('  ',hit['citation'],'score',hit['score'])
        print('    ',hit['excerpt'][:220].replace('\n',' '))
assert course_eval['all_returned_citations_resolve']


ModuleNotFoundError: No module named 'nbformat'